In [12]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [10]:
def build_models_adaptation():
    models = {
        "linreg": LinearRegression(),
        "boost": LGBMRegressor(
            n_estimators=800,
            learning_rate=0.02,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            verbose=-1),
        "pls": PLSRegression(n_components=35,
            scale=True,
            max_iter=2000)}
    return models

def adapt_ace_to_discover(ace_df, disc_df, all_targets, split_date="2021-01-01", adapt_model_name="linreg"):
    """
    Адаптация ace -> discover

    1. K обучается только на пересечении тренировочных данных ace и discover
    2. Ace до появления discover адаптируется в домен discover
    3. Возвращаются адаптированный ace_old_train и discr_train, disc_test
    """

    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]

    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in all_targets]

    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])

    # Модель адаптации K: ace -> discover
    models = build_models_adaptation()
    
    if adapt_model_name not in models:
        raise ValueError(f"Неизвестная модель: {adapt_model_name}. Доступны: {list(models.keys())}")
    
    K = models[adapt_model_name]

    # Используем, так как таргет многомерный
    if adapt_model_name == "boost":
        from sklearn.multioutput import MultiOutputRegressor
        K = MultiOutputRegressor(K)
        
    K.fit(X_ace, X_disc)

    # ace до появления discover
    ace_old = ace.loc[:disc.index.min()]
    ace_old = ace_old[feature_cols]

    X_ace_old = sc_ace.transform(ace_old)
    X_ace_old_adapted = K.predict(X_ace_old)
    X_ace_old_adapted = sc_disc.inverse_transform(X_ace_old_adapted)

    ace_old_adapted = pd.DataFrame(X_ace_old_adapted, index=ace_old.index, columns=feature_cols)

    for col in all_targets:
        ace_old_adapted[col] = ace_df.loc[ace_old.index, col]

    return ace_old_adapted, disc_train, disc_test, K, sc_ace, sc_disc

In [21]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
        n_estimators=700,
        learning_rate=0.02,
        max_depth=-1, #5,
        num_leaves=30,
        subsample=0.9,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        random_state=random_state,
        verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
        hidden_layer_sizes=(256, 256, 128),
        activation='tanh', # relu ok, работает быстрее
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        early_stopping=True,
        validation_fraction=0.1,
        max_iter=1500,
        random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_D(ace_old_adapted, disc_train, disc_test, split_date, all_targets, target, adaptation_method='linreg', delays='24h', results_list=None):
    """
    Обучение M_D:
    - train = адаптированный старый ace + discover_train
    - test = discover_test
    """

    train_adapt = pd.concat([ace_old_adapted, disc_train], axis=0).sort_index()

    feature_cols = [c for c in train_adapt.columns if c not in all_targets]

    X_train = train_adapt[feature_cols].values
    y_train = train_adapt[target].values

    X_test = disc_test[feature_cols].values
    y_test = disc_test[target].values
    
    results = {}
    results[target] = {}

    for name, model in models.items():
        if name == 'LGBM':
            # Для бустинга выделяем валидационный набор
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train, test_size=0.1, random_state=42)

            model.fit(X_tr, y_tr,
                boost__eval_set=[(X_val, y_val)],
                boost__eval_metric='l2')
        else:
            model.fit(X_train, y_train)
            
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        if results_list is None:
            results_list = []
            
        results_list.append({
            'Target': target,
            'Delays': delays,
            'Adaptation_method': adaptation_method,
            'Forecast_model': name,
            'RMSE': rmse,
            'R2': r2
        })
        print(f"{name}: rmse={rmse:.4f}, r2={r2:.4f}")
        
    return results

In [5]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2021-01-01"
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3']

In [15]:
# 2.1. Доменная адаптация с помощью моделей линейной регрессии (linreg), градиентного бустинга (boost) и метода проекций на латентные структуры (pls)
ace_adapted_lin_24, disc_train_24, disc_test_24, K_lin_24, sd_lin_24, sa_lin_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="linreg"
)

ace_adapted_gbr_24, disc_train_24, disc_test_24, K_gbr_24, sd_gbr_24, sa_gbr_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="boost"
)

ace_adapted_pls_24, disc_train_24, disc_test_24, K_pls_24, sd_pls_24, sa_pls_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="pls"
)

In [17]:
# ace_old_adapted_af, disc_train_af, disc_test_af, K_af, sc_ace_af, sc_disc_af \
# = adapt_ace_to_discover(data_ace_af_copy, data_discover_af_copy, targets, split_date)

In [22]:
results_list = []

print(f"\n==== Depth - 24h ====")

print(f"\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_lin_24, disc_train_24, disc_test_24, split_date, targets, target_col, adaptation_method='linreg', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_gbr_24, disc_train_24, disc_test_24, split_date, targets, target_col, adaptation_method='lgbm', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_pls_24, disc_train_24, disc_test_24, split_date, targets, target_col, adaptation_method='pls', delays='24h', results_list=results_list)


==== Depth - 24h ====

==== Adaptation - linreg ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=3.2704, r2=0.9636
Ridge: rmse=3.2702, r2=0.9636
Lasso: rmse=3.2691, r2=0.9636
LGBM: rmse=3.4452, r2=0.9596
MLP: rmse=4.4292, r2=0.9332

==== Forecast of DST_PLUS2 ====
Linear: rmse=5.0843, r2=0.9119
Ridge: rmse=5.0841, r2=0.9119
Lasso: rmse=5.0796, r2=0.9121
LGBM: rmse=5.0266, r2=0.9139
MLP: rmse=6.6770, r2=0.8481

==== Forecast of DST_PLUS3 ====
Linear: rmse=6.4743, r2=0.8571
Ridge: rmse=6.4743, r2=0.8571
Lasso: rmse=6.4702, r2=0.8573
LGBM: rmse=6.3305, r2=0.8634
MLP: rmse=8.3205, r2=0.7640

==== Adaptation - lgbm ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=3.3746, r2=0.9612
Ridge: rmse=3.3744, r2=0.9612
Lasso: rmse=3.3628, r2=0.9615
LGBM: rmse=3.4156, r2=0.9603
MLP: rmse=4.5292, r2=0.9301

==== Forecast of DST_PLUS2 ====
Linear: rmse=5.1453, r2=0.9098
Ridge: rmse=5.1452, r2=0.9098
Lasso: rmse=5.1391, r2=0.9100
LGBM: rmse=4.9881, r2=0.9152
MLP: rmse=6.7815, r2=0.8433

==== Foreca

In [23]:
results_df = pd.DataFrame(results_list)

In [24]:
results_df

,Target,Delays,Adaptation_method,Forecast_model,RMSE,R2
0,Dst_plus1,24h,linreg,Linear,3.270406,0.963573
1,Dst_plus1,24h,linreg,Ridge,3.270220,0.963577
2,Dst_plus1,24h,linreg,Lasso,3.269071,0.963603
3,Dst_plus1,24h,linreg,LGBM,3.445193,0.959575
4,Dst_plus1,24h,linreg,MLP,4.429175,0.933186
5,Dst_plus2,24h,linreg,Linear,5.084258,0.911931
6,Dst_plus2,24h,linreg,Ridge,5.084105,0.911936
7,Dst_plus2,24h,linreg,Lasso,5.079568,0.912093
8,Dst_plus2,24h,linreg,LGBM,5.026647,0.913916
9,Dst_plus2,24h,linreg,MLP,6.677025,0.848108


In [25]:
results_df.to_excel("results/models-adaptation-ace-to-discover.xlsx", index=False)